# Monte Carlo and yield

A campaign without faults: the spread of the healthy circuit, and how many meet their
specifications. Also runnable as `python examples/08_monte_carlo.py`.

In [1]:

import sys
from pathlib import Path

import pandas as pd

examples_dir = next(
    (root / "examples" for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "examples" / "rc_study.py").is_file()),
    None,
)
if examples_dir is not None:
    sys.path.insert(0, str(examples_dir))

In [2]:
from rc_study import OUTPUT, WORKERS, circuit, config, measurements, population  # noqa: E402

from spicefault import Campaign, Specification  # noqa: E402
from spicefault.statistics import samples_for_half_width, zero_failure_bound  # noqa: E402

## Draw and simulate the healthy population

No fault is given: every sample is a circuit drawn from the tolerances.

In [3]:
if __name__ == "__main__":
    pd.set_option("display.width", 160)
    pd.set_option("display.float_format", "{:.4g}".format)
    campaign = Campaign(
        circuit,
        out_dir=OUTPUT / "rc_monte_carlo",
        samples=1000,
        variations=population,
        config=config,
        measurements=measurements,
        seed=42,
    )
    campaign.run(workers=WORKERS, chunk=250, progress=False)
    dataset = campaign.dataset()
    print(dataset)
    print("problems found by verify():", dataset.verify())

Dataset('examples/output/rc_monte_carlo': 1000 samples, 0 faults, 5 measurements)
problems found by verify(): []


## The distribution of each measurement

A quantile that the sample cannot support is left empty instead of being read off the
smallest or the largest value.

In [4]:
if __name__ == "__main__":
    statistics = dataset.statistics(quantiles=(0.001, 0.05, 0.5, 0.95))
    print("\nmean, spread and failed simulations:")
    print(statistics[["n", "n_failed", "mean", "std", "minimum", "maximum"]].droplevel([0, 1]))
    print("\nquantiles, with their 95 % intervals:")
    print(statistics[["q0.001", "q0.05_low", "q0.05", "q0.05_high", "q0.5"]].droplevel([0, 1]))


mean, spread and failed simulations:
                n  n_failed   mean      std  minimum  maximum
measurement                                                  
dc           1000         0    0.5 0.002082   0.4951   0.5048
gain_100     1000         0  0.477 0.002333   0.4705   0.4833
gain_1k      1000         0 0.1518 0.004062    0.144   0.1603
phase_1k     1000         0 -72.32   0.4834    -73.2   -71.41
peak         1000         0    0.5 0.002082    0.495   0.5048

quantiles, with their 95 % intervals:
             q0.001  q0.05_low  q0.05  q0.05_high   q0.5
measurement                                             
dc              NaN     0.4961 0.4963      0.4966    0.5
gain_100        NaN     0.4729 0.4732      0.4735  0.477
gain_1k         NaN     0.1454 0.1457       0.146 0.1516
phase_1k        NaN     -73.08 -73.05      -73.03 -72.33
peak            NaN     0.4961 0.4963      0.4966    0.5


## Yield against specification limits

The limits are kept apart from the simulation: changing them needs no new simulation.

In [5]:
if __name__ == "__main__":
    limits = [
        Specification("dc", minimum=0.4965, maximum=0.5035),
        Specification("gain_1k", minimum=0.146),
        Specification("phase_1k", minimum=-73.1),
    ]
    report = dataset.yield_report(limits)
    print("\nyield per specification and of all of them:")
    print(report[["n", "passed", "yield", "ci_low", "ci_high", "only_this", "margin_sigma"]])

    relaxed = [Specification("dc", minimum=0.495, maximum=0.505), *limits[1:]]
    print(f"\nwith the DC limits at +-0.005: yield "
          f"{dataset.yield_report(relaxed).loc['all', 'yield']:.3f}")  # fmt: skip


yield per specification and of all of them:
                  n  passed  yield  ci_low  ci_high  only_this  margin_sigma
specification                                                               
dc             1000     898  0.898  0.8777   0.9153         94         1.669
gain_1k        1000     934  0.934  0.9169   0.9478         36         1.431
phase_1k       1000     972  0.972  0.9598   0.9806          3         1.608
all            1000     836  0.836  0.8118   0.8577        NaN           NaN

with the DC limits at +-0.005: yield 0.930


## What the sample size allows

In [6]:
if __name__ == "__main__":
    wide = dataset.yield_report([Specification("dc", minimum=0.4, maximum=0.6)]).loc["all"]
    print(f"\nno failure in {wide['n']:.0f} circuits: the failure probability is below "
          f"{zero_failure_bound(int(wide['n'])):.4f}, not zero")  # fmt: skip
    print("circuits for a yield near 0.95 within +-0.005:",
          samples_for_half_width(0.005, p=0.95))  # fmt: skip


no failure in 1000 circuits: the failure probability is below 0.0030, not zero
circuits for a yield near 0.95 within +-0.005: 7299
